# Simglucose Simulation Results Notebook

## Instructions

[Comprehensive documentation here](Docs/simglucose.md) detailing how to run Trio oref algorithm variants through simglucose to conduct mechanistic in silico simulations.

### Run One Virtual Person

```
python3 simglucose/run_sim.py -u <virtual_patient> -a <alg_name> -d <days> -scen <meal_scenario_path> -fn <results_file>
```
* `virtual_patient`: Name of virtual patient. Valid patient names are age group followed by three digits. Age groups = [child, adolescent, adult]. Valid digits = [001, 002, 003, 004, 005, 006, 007, 008, 009, 010]
    * eg. adolescent002 or adult010
* `alg_name`: Optional argument. Name of the Javascript oref algorithm variant to run instead of the Swift algorithm. Defaults to `"swift"` if no argument given. Possible choices:
    * `jsbug`: Original Javascript implementation.
    * `js`: Javascript implementation of bug-free Swift oref algorithm.
    * `swift`: Swift implementation of oref algorithm.
* `days`: Number of days simulation runs (must be whole number)
* `meal_scenario_path`: Optional argument. Filepath to precomputed .npy file containing meal scenario.
* `results_file`: Filepath to csv file where outputs will be written to.

### Run All Virtual People
Run the following commands to execute this [script](./Scripts/run_simglucose.sh) which simulates all 30 virtual persons. There is the option to run simulations in parallel in independent processes. Set `PARALLELISM` to the number of processes you want to run concurrently. 

```shell
chmod +x Scripts/run_simglucose.sh # if have not run script yet
./Scripts/run_simglucose.sh
```

## Code

In [1]:
import numpy as np
import pandas as pd

In [2]:
children = [f"child{n}" for n in ["001","002","003","004","005","006","007","008","009","010"]]
children_display = [f"child {n}" for n in range(1, 11)]
adolescents = [f"adolescent{n}" for n in ["001","002","003","004","005","006","007","008","009","010"]]
adolescents_display = [f"adolescent {n}" for n in range(1, 11)]
adults = [f"adult{n}" for n in ["001","002","003","004","005","006","007","008","009","010"]]
adults_display = [f"adult {n}" for n in range(1, 11)]
all_users = children + adolescents + adults
all_users_display = children_display + adolescents_display + adults_display

results_folder = "simglucoseResults"

In [3]:
def glycemia_risk_index(tar, tbr, tvar, tvbr):
    gri = 3 * tvbr + 2.4 * tbr + 1.6 * tvar + 0.8 * tar
    return 100.0 if gri > 100.0 else gri

def glucose_stats(glucose):
    low_bound = 70
    v_low_bound = 54
    v_high_bound = 250
    high_bound = 180
    glucose = np.array(glucose)

    tar = 100 * np.average(glucose > high_bound)
    tvar = 100 * np.average(glucose > v_high_bound)
    tbr = 100 * np.average(glucose < low_bound)
    tvbr = 100 * np.average(glucose < v_low_bound)
    tir = 100 - tar - tbr

    gri = glycemia_risk_index(tar, tbr, tvar, tvbr)

    return tir, tar, tbr, tvar, tvbr, gri

In [4]:
def user_results(user:str, alg:str):
    path = f"{results_folder}/{user}/{alg}.csv"
    glucose = pd.read_csv(path)['CGM'].to_list()
    return glucose_stats(glucose)

def age_group_results(group:list, user_display:list, alg:str):
    results = {"User":[], "TIR":[], "TAR":[], "TBR":[], "TVAR":[], 
                     "TVBR":[], "GRI":[]}
    
    for i in range(len(group)):
        tir, tar, tbr, tvar, tvbr, gri = user_results(group[i], alg)
        results["User"].append(user_display[i])
        results["TIR"].append(tir)
        results["TAR"].append(tar)
        results["TBR"].append(tbr)
        results["TVAR"].append(tvar)
        results["TVBR"].append(tvbr)
        results["GRI"].append(gri)

    for category, values in results.items():
        if category == "User": 
            results["User"].append("average")
            results["User"].append("standard deviation")
        else:
            avg = np.average(results[category]).item()
            sd = np.std(results[category]).item()
            results[category].extend([avg, sd])

    return results

In [5]:
print('Children Swift oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(children, children_display, 'swift'))
display(df.style.hide())

print('Children Buggy Javascript oref Simlation Results')
df = pd.DataFrame.from_dict(age_group_results(children, children_display, 'jsbug'))
display(df.style.hide())

Children Swift oref Simulation Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
child 1,78.923878,15.794694,5.281428,4.215224,1.090999,35.328540
child 2,78.799901,16.761716,4.438383,0.123977,0.818249,26.714605
child 3,67.443590,26.729482,5.826928,8.678403,0.793454,51.634019
child 4,84.428465,13.141582,2.429953,0.520704,0.297545,18.070915
child 5,89.437144,6.124473,4.438383,0.049591,0.669477,17.639474
child 6,77.584924,20.332259,2.082817,4.934292,0.123977,29.531366
child 7,92.511778,5.207042,2.281180,0.074386,0.247954,10.503347
child 8,46.441855,44.135879,9.422266,31.961319,4.661542,100.000000
child 9,89.164394,9.199107,1.636499,0.223159,0.123977,12.015869
child 10,70.964543,27.002232,2.033226,12.571287,0.347136,47.636995


Children Buggy Javascript oref Simlation Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
child 1,78.725515,15.571535,5.702951,4.215224,1.586908,37.649393
child 2,80.262832,15.075626,4.661542,0.123977,0.793454,25.826928
child 3,66.625341,26.580709,6.793950,8.604017,0.545500,52.972973
child 4,83.684602,12.769650,3.545748,0.892636,0.471113,21.567072
child 5,88.792462,6.446814,4.760724,0.049591,0.892636,19.340441
child 6,77.312175,20.307463,2.380362,4.934292,0.297545,30.746343
child 7,94.073890,4.116043,1.810067,0.049591,0.322341,8.683362
child 8,46.863377,44.036697,9.099926,32.234069,4.711133,100.000000
child 9,88.693280,9.397471,1.909249,0.247954,0.148773,12.943218
child 10,71.510042,26.903050,1.586908,12.149764,0.148773,45.216960


In [6]:
print('Adolescent Swift oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(adolescents, adolescents_display, 'swift'))
display(df.style.hide())

print('Adolescent Buggy Javascript oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(adolescents, adolescents_display, 'jsbug'))
display(df.style.hide())

Adolescent Swift oref Simulation Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
adolescent 1,99.256137,0.074386,0.669477,0.000000,0.000000,1.666253
adolescent 2,63.352343,34.168113,2.479544,4.611951,0.173568,41.185222
adolescent 3,92.263823,4.016861,3.719316,0.000000,0.595091,13.925118
adolescent 4,87.106372,6.719564,6.174064,0.000000,1.041408,23.317630
adolescent 5,77.684106,17.902306,4.413588,1.983635,0.942227,30.914952
adolescent 6,93.230846,5.628564,1.140590,0.024795,0.223159,7.949417
adolescent 7,69.625589,27.299777,3.074634,5.926110,0.495909,40.188445
adolescent 8,72.278701,24.795438,2.925862,3.471361,0.024795,32.486982
adolescent 9,94.644185,4.116043,1.239772,0.000000,0.049591,6.417059
adolescent 10,91.346392,4.116043,4.537565,0.000000,0.595091,15.968262


Adolescent Buggy Javascript oref Simulation Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
adolescent 1,99.156955,0.074386,0.768659,0.000000,0.000000,1.904290
adolescent 2,61.715844,35.755021,2.529135,4.487974,0.272750,42.672948
adolescent 3,92.958096,3.868088,3.173816,0.000000,0.123977,11.083561
adolescent 4,87.999008,6.793950,5.207042,0.000000,0.793454,20.312423
adolescent 5,77.336970,17.951897,4.711133,1.760476,0.768659,30.790974
adolescent 6,92.908505,5.826928,1.264567,0.049591,0.223159,8.445326
adolescent 7,70.146293,27.101413,2.752294,5.950905,0.495909,39.295810
adolescent 8,72.824200,24.993801,2.181999,3.396975,0.049591,30.815770
adolescent 9,94.594595,3.942475,1.462931,0.000000,0.000000,6.665014
adolescent 10,91.519960,3.967270,4.512770,0.000000,0.793454,16.384825


In [7]:
print('Adult Swift oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(adults, adults_display, 'swift'))
display(df.style.hide())

print('Adult Buggy Javascript oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(adults, adults_display, 'jsbug'))
display(df.style.hide())

Adult Swift oref Simulation Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
adult 1,92.164642,2.702703,5.132656,0.000000,0.495909,15.968262
adult 2,97.842797,0.396727,1.760476,0.000000,0.099182,4.840069
adult 3,95.933548,0.173568,3.892884,0.000000,0.942227,12.308455
adult 4,92.015869,6.347632,1.636499,0.099182,0.148773,9.610712
adult 5,97.570047,1.066204,1.363749,0.000000,0.000000,4.125961
adult 6,87.379122,2.752294,9.868584,0.000000,1.165386,29.382594
adult 7,94.545004,0.991818,4.463179,0.000000,0.570295,13.215968
adult 8,97.545252,0.099182,2.355567,0.000000,0.297545,6.625341
adult 9,82.420035,8.281676,9.298289,0.198364,1.760476,34.540045
adult 10,94.073890,1.562113,4.363997,0.000000,0.297545,12.615919


Adult Buggy Javascript oref Simulation Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
adult 1,91.941483,2.777089,5.281428,0.000000,0.446318,16.236053
adult 2,97.446070,0.396727,2.157203,0.000000,0.099182,5.792214
adult 3,95.561617,0.198364,4.240020,0.000000,0.595091,12.120010
adult 4,92.536573,5.926110,1.537317,0.099182,0.173568,9.109844
adult 5,97.669229,0.991818,1.338954,0.000000,0.148773,4.453261
adult 6,87.949417,2.504339,9.546243,0.000000,1.710885,30.047111
adult 7,94.445822,1.041408,4.512770,0.000000,0.570295,13.374659
adult 8,97.371684,0.148773,2.479544,0.000000,0.272750,6.888173
adult 9,81.973717,8.777585,9.248698,0.173568,1.810067,34.926853
adult 10,93.330027,1.413340,5.256633,0.000000,0.471113,15.159931


In [8]:
print('All Virtual Persons Swift oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(all_users, all_users_display, 'swift'))
display(df.style.hide())

print('All Virtual Persons Buggy Javascript oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(all_users, all_users_display, 'jsbug'))
display(df.style.hide())

All Virtual Persons Swift oref Simulation Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
child 1,78.923878,15.794694,5.281428,4.215224,1.090999,35.328540
child 2,78.799901,16.761716,4.438383,0.123977,0.818249,26.714605
child 3,67.443590,26.729482,5.826928,8.678403,0.793454,51.634019
child 4,84.428465,13.141582,2.429953,0.520704,0.297545,18.070915
child 5,89.437144,6.124473,4.438383,0.049591,0.669477,17.639474
child 6,77.584924,20.332259,2.082817,4.934292,0.123977,29.531366
child 7,92.511778,5.207042,2.281180,0.074386,0.247954,10.503347
child 8,46.441855,44.135879,9.422266,31.961319,4.661542,100.000000
child 9,89.164394,9.199107,1.636499,0.223159,0.123977,12.015869
child 10,70.964543,27.002232,2.033226,12.571287,0.347136,47.636995


All Virtual Persons Buggy Javascript oref Simulation Results


User,TIR,TAR,TBR,TVAR,TVBR,GRI
child 1,78.725515,15.571535,5.702951,4.215224,1.586908,37.649393
child 2,80.262832,15.075626,4.661542,0.123977,0.793454,25.826928
child 3,66.625341,26.580709,6.793950,8.604017,0.545500,52.972973
child 4,83.684602,12.769650,3.545748,0.892636,0.471113,21.567072
child 5,88.792462,6.446814,4.760724,0.049591,0.892636,19.340441
child 6,77.312175,20.307463,2.380362,4.934292,0.297545,30.746343
child 7,94.073890,4.116043,1.810067,0.049591,0.322341,8.683362
child 8,46.863377,44.036697,9.099926,32.234069,4.711133,100.000000
child 9,88.693280,9.397471,1.909249,0.247954,0.148773,12.943218
child 10,71.510042,26.903050,1.586908,12.149764,0.148773,45.216960


## Parkes Error Analysis

In [9]:
from ParkesErrorGrid.parkes_error import ParkesError

In [10]:
def user_parkes_grid(user, user_display=None, gen_plot=False):

    if gen_plot and not user_display:
        raise Exception(f"Must give value for user_display if gen_plot=True.")
    
    ref_trace = pd.read_csv(f"simglucoseResults/{user}/swift.csv")["CGM"].to_list()
    pred_trace = pd.read_csv(f"simglucoseResults/{user}/jsbug.csv")["CGM"].to_list()
    grid = ParkesError(ref_trace, pred_trace)
    if gen_plot:
        grid.plot(user_display, "Swift Algorithm Glucose", 
                "Original Javascript Algorithm Glucose", size=1, 
                save_fig_path=f"simglucoseResults/{user}/parkes_error.png")
        
    grid_flip = ParkesError(pred_trace, ref_trace)
    if gen_plot:
        grid_flip.plot(user_display, "Original Javascript Algorithm Glucose",
                "Swift Algorithm Glucose", size=1, 
                save_fig_path=f"simglucoseResults/{user}/parkes_error_flipped.png")

    return grid.zone_count(), grid_flip.zone_count()


In [11]:
swift_ref_users = {}
js_ref_users = {}

for user in all_users:
    swift_ref, js_ref = user_parkes_grid(user)
    n = sum(swift_ref.values())
    swift_ref_users[user] = [round(100*count/n, 2) for count in swift_ref.values()]
    js_ref_users[user] = [round(100*count/n, 2) for count in js_ref.values()]

In [12]:
df = pd.DataFrame.from_dict(swift_ref_users, orient='index', columns=['A', 'B', 'C', 'D', 'E'])
display(df)

,A,B,C,D,E
child001,97.59,2.41,0.00,0.0,0.0
child002,99.98,0.02,0.00,0.0,0.0
child003,100.00,0.00,0.00,0.0,0.0
child004,91.97,7.79,0.25,0.0,0.0
child005,99.83,0.17,0.00,0.0,0.0
child006,100.00,0.00,0.00,0.0,0.0
child007,100.00,0.00,0.00,0.0,0.0
child008,99.88,0.12,0.00,0.0,0.0
child009,99.43,0.57,0.00,0.0,0.0
child010,99.85,0.15,0.00,0.0,0.0


In [13]:
df = pd.DataFrame.from_dict(js_ref_users, orient='index', columns=['A', 'B', 'C', 'D', 'E'])
display(df)

,A,B,C,D,E
child001,98.86,1.14,0.0,0.0,0.0
child002,100.00,0.00,0.0,0.0,0.0
child003,99.88,0.12,0.0,0.0,0.0
child004,92.64,7.36,0.0,0.0,0.0
child005,100.00,0.00,0.0,0.0,0.0
child006,100.00,0.00,0.0,0.0,0.0
child007,100.00,0.00,0.0,0.0,0.0
child008,99.78,0.22,0.0,0.0,0.0
child009,99.50,0.50,0.0,0.0,0.0
child010,100.00,0.00,0.0,0.0,0.0
